In [1]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 9장. 회귀 분석으로 숫자 예측하기\n",
    "\n",
    "이 Notebook의 핵심은 높은 점수를 만드는 것이 아니라 다음 순서를 지키는 것입니다.\n",
    "\n",
    "`예측 시점 → 누수 방지 → 시간 분할 → Train TimeSeriesSplit → 모델 선택 고정 → Final Test → Baseline 비교`\n",
    "\n",
    "**Final Test 결과를 보고 모델을 고르지 않습니다.**"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 0. 실행 전 확인\n",
    "\n",
    "Chapter09는 Chapter05의 오류 탐지용 전용 Raw가 아니라 **공통 프로젝트 `data/raw`**를 사용합니다. 먼저 프로젝트 루트에서 다음 명령을 실행해 관계 검증을 통과한 모델링 입력을 준비합니다.\n",
    "\n",
    "```powershell\n",
    "python scripts/prepare_ch09_data.py\n",
    "```\n",
    "\n",
    "이 명령은 공통 `data/raw`를 전처리하고 FK 관계를 검증한 뒤 `data/processed/*_clean.csv`를 생성합니다."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from pathlib import Path\n",
    "import sys\n",
    "import pandas as pd\n",
    "\n",
    "CURRENT_DIR = Path.cwd().resolve()\n",
    "PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR\n",
    "if str(PROJECT_ROOT) not in sys.path:\n",
    "    sys.path.insert(0, str(PROJECT_ROOT))\n",
    "\n",
    "PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'\n",
    "REPORT_DIR = PROJECT_ROOT / 'reports'\n",
    "\n",
    "from src.regression import (\n",
    "    FEATURE_COLUMNS,\n",
    "    FORBIDDEN_FEATURES,\n",
    "    TARGET_COLUMN,\n",
    "    build_feature_audit,\n",
    "    build_leakage_checklist,\n",
    "    build_regression_dataset,\n",
    "    build_regression_validation,\n",
    "    build_split_summary,\n",
    "    create_prediction_result,\n",
    "    cross_validate_regression_models,\n",
    "    load_regression_source_data,\n",
    "    make_regression_models,\n",
    "    run_regression_analysis,\n",
    "    save_regression_outputs,\n",
    "    select_diagnostic_model,\n",
    "    split_model_data_by_time,\n",
    "    train_and_evaluate_models,\n",
    "    validate_feature_columns,\n",
    ")\n",
    "\n",
    "print('PROJECT_ROOT:', PROJECT_ROOT)\n",
    "print('PROCESSED_DIR:', PROCESSED_DIR)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. 예측 문제와 예측 시점\n",
    "\n",
    "- Target: 주문 1건의 주문 상세 금액 합계 `order_total`\n",
    "- 예측 시점: 주문 메타데이터와 고객의 비식별 특성은 알 수 있지만 주문 상세 수량·단가·금액은 모델에 제공하지 않는 시점\n",
    "- 목표 계산 재료와 사후 정보, 식별자는 feature에서 제외합니다."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print('Target:', TARGET_COLUMN)\n",
    "print('Allowed features:', FEATURE_COLUMNS)\n",
    "print('Forbidden features:', sorted(FORBIDDEN_FEATURES))\n",
    "validate_feature_columns()\n",
    "print('Feature leakage gate: PASS')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 2. 검증된 전처리 데이터를 불러옵니다"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "data = load_regression_source_data(PROCESSED_DIR)\n",
    "for name, frame in data.items():\n",
    "    print(name, frame.shape)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. 주문 단위 Target을 만들고 관계를 검증합니다\n",
    "\n",
    "`line_total = quantity × unit_price`를 확인하고, 주문과 Target은 `one_to_one`, 주문과 고객은 `many_to_one` 관계가 깨지지 않는지 확인합니다."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "model_data = build_regression_dataset(\n",
    "    customers=data['customers'],\n",
    "    orders=data['orders'],\n",
    "    order_items=data['order_items'],\n",
    ")\n",
    "display(model_data.head())\n",
    "print('Model rows:', len(model_data))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 4. Feature Audit을 확인합니다"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "feature_audit = build_feature_audit()\n",
    "display(feature_audit)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. 시간 순서로 Train / Final Test를 분리합니다\n",
    "\n",
    "같은 달력 날짜를 양쪽에 나누지 않습니다. Final Test는 모델 선택이 끝날 때까지 사용하지 않습니다."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "train_data, test_data = split_model_data_by_time(model_data, test_size=0.2)\n",
    "split_summary = build_split_summary(train_data, test_data)\n",
    "display(split_summary)\n",
    "assert train_data['order_date'].max().normalize() < test_data['order_date'].min().normalize()\n",
    "print('Chronological split gate: PASS')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "X_train = train_data[FEATURE_COLUMNS].copy()\n",
    "X_test = test_data[FEATURE_COLUMNS].copy()\n",
    "y_train = train_data[TARGET_COLUMN].copy()\n",
    "y_test = test_data[TARGET_COLUMN].copy()\n",
    "print('Train:', X_train.shape, 'Final Test:', X_test.shape)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Pipeline 후보 모델을 준비합니다\n",
    "\n",
    "숫자형 결측 대체·표준화와 범주형 결측 대체·One-Hot Encoding은 모델 Pipeline 안에서 각 Train Fold로만 학습됩니다."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "models = make_regression_models(random_state=42)\n",
    "print(list(models))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Train 기간의 TimeSeriesSplit으로 후보를 비교합니다\n",
    "\n",
    "여기까지는 Final Test의 `y_test`를 모델 선택에 사용하지 않습니다."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "cv_summary = cross_validate_regression_models(\n",
    "    models=models,\n",
    "    X_train=X_train,\n",
    "    y_train=y_train,\n",
    ")\n",
    "display(cv_summary)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. 비베이스라인 모델 선택을 고정합니다\n",
    "\n",
    "Linear Regression과 Random Forest 중 Train CV의 평균 MAE가 더 낮은 모델을 선택합니다. **이 시점까지 Final Test 성능은 보지 않습니다.**"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "selected_model_name = select_diagnostic_model(cv_summary)\n",
    "print('Frozen selected model:', selected_model_name)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. 이제 Final Test에서 Baseline과 고정 모델만 평가합니다\n",
    "\n",
    "Final Test 결과가 기대와 다르더라도 같은 Test를 보고 다른 후보로 교체하지 않습니다."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "model_comparison, predictions = train_and_evaluate_models(\n",
    "    models=models,\n",
    "    selected_model_name=selected_model_name,\n",
    "    X_train=X_train,\n",
    "    X_test=X_test,\n",
    "    y_train=y_train,\n",
    "    y_test=y_test,\n",
    ")\n",
    "display(model_comparison)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 지표 읽기\n",
    "\n",
    "- MAE: 평균적으로 얼마만큼 틀리는가\n",
    "- RMSE: 큰 오차에 더 민감\n",
    "- R²: 평균 예측 대비 설명력, 음수가 될 수 있음\n",
    "- MAE 개선율: Dummy baseline 대비 고정 모델의 Final Test MAE 개선 정도"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 10. 실제값·예측값·잔차를 진단합니다"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "prediction_result = create_prediction_result(\n",
    "    test_data=test_data,\n",
    "    y_test=y_test,\n",
    "    y_pred=predictions[selected_model_name],\n",
    "    model_name=selected_model_name,\n",
    ")\n",
    "display(prediction_result.head(10))\n",
    "print('주의: order_id가 포함된 이 표는 내부 진단용입니다.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 11. 자동 Validation Evidence를 만듭니다"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "validation = build_regression_validation(\n",
    "    train_data=train_data,\n",
    "    test_data=test_data,\n",
    "    cv_summary=cv_summary,\n",
    "    selected_model_name=selected_model_name,\n",
    "    model_comparison=model_comparison,\n",
    ")\n",
    "display(validation)\n",
    "assert validation['status'].eq('PASS').all()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 12. Evidence·내부 진단·공개 보고서를 구분해 저장합니다"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "checklist = build_leakage_checklist()\n",
    "output_paths = save_regression_outputs(\n",
    "    model_data=model_data,\n",
    "    split_summary=split_summary,\n",
    "    feature_audit=feature_audit,\n",
    "    cv_summary=cv_summary,\n",
    "    selected_model_name=selected_model_name,\n",
    "    model_comparison=model_comparison,\n",
    "    prediction_result=prediction_result,\n",
    "    validation=validation,\n",
    "    checklist=checklist,\n",
    "    report_dir=REPORT_DIR,\n",
    ")\n",
    "for name, path in output_paths.items():\n",
    "    print(name, '->', path)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 13. 전체 공통 함수를 새 실행으로 다시 확인합니다\n",
    "\n",
    "셀의 메모리 상태가 아니라 공통 함수가 같은 순서를 재현하는지 확인합니다."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "result = run_regression_analysis(\n",
    "    processed_dir=PROCESSED_DIR,\n",
    "    report_dir=REPORT_DIR,\n",
    "    test_size=0.2,\n",
    "    random_state=42,\n",
    ")\n",
    "print('Selected by Train CV:', result['selected_model_name'])\n",
    "display(result['validation'])"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 14. 해석\n",
    "\n",
    "낮은 성능이나 음수 R²도 숨기지 않습니다. 현재 예측 시점에 허용한 정보가 약하다면 `현재 데이터로는 모델 사용 보류`도 올바른 결론입니다.\n",
    "\n",
    "LLM이 만든 코드도 같은 예측 시점·누수·Pipeline·Train CV·Final Test 보호 기준으로 검토합니다."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 15. 완료 체크\n",
    "\n",
    "- [ ] `python scripts/prepare_ch09_data.py`로 공통 모델링 입력을 준비했다.\n",
    "- [ ] Target과 예측 시점을 설명할 수 있다.\n",
    "- [ ] 금지 feature가 입력에 없음을 확인했다.\n",
    "- [ ] 같은 날짜가 Train과 Final Test에 동시에 들어가지 않는다.\n",
    "- [ ] 전처리는 Pipeline 내부에서 학습된다.\n",
    "- [ ] Train TimeSeriesSplit으로 후보를 비교했다.\n",
    "- [ ] Final Test 전에 Selected Model을 고정했다.\n",
    "- [ ] Final Test에서는 Baseline과 Frozen Model만 비교했다.\n",
    "- [ ] MAE, RMSE, R²를 함께 읽었다.\n",
    "- [ ] 내부 식별자 결과와 공개 보고서를 구분했다.\n",
    "- [ ] Validation Evidence가 모두 PASS다."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

NameError: name 'null' is not defined